In [1]:
testrun = "false"
# testrun = "true"
version = "v1"

In [2]:
print(testrun)

false


# Evaluate with GPT-4

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import pickle


In [5]:
import re
import openai
import pandas as pd
import random


import sys
import os
import tiktoken
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
)  # for exponential backoff

In [6]:
from langchain_openai import AzureChatOpenAI
	
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.memory import ChatMessageHistory

from langchain.prompts.chat import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain.prompts.chat import ChatPromptTemplate, HumanMessagePromptTemplate

In [7]:
from ebell_llm_funcs import ebell_CAP_prob_llm

In [8]:
%load_ext dotenv
%dotenv /vast/palmer/home.mccleary/vs428/Documents/DischargeMe/hail-dischargeme/.env

In [9]:
# openai.api_type = "azure"
engine = "decile-gpt-4-128K"

In [10]:
os.environ['AZURE_OPENAI_ENDPOINT'] = os.getenv("AZURE_OPENAI_ENDPOINT")
os.environ['AZURE_OPENAI_API_KEY'] = os.getenv("AZURE_OPENAI_KEY")
os.environ["OPENAI_API_VERSION"] = "2023-12-01-preview"

In [11]:
llm = AzureChatOpenAI(
    deployment_name=engine
)

In [12]:
def num_tokens_from_string(string: str, encoding_name: str="cl100k_base") -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

In [13]:
from IPython.display import display, HTML

def add_line_breaks(text):
    return text.replace('\n', '<br>')


def pretty_print(df):
    return display( HTML( df.to_html().replace("\\n","<br>") ) )

In [14]:
cases = pd.read_csv("/home/vs428/project/Uncertainty_data/penumonia_all_cases.csv", sep="|")

In [15]:
with pd.option_context('display.max_colwidth', None):
    display(cases)

,note_id,scenario,probability_q,management_q,valupdate_management_q,management_option_1,management_option_2,management_option_3,true_pretest
0,1,"A patient presents with a 3 day history of cough productive of yellowish green sputum and a runny nose. He is afebrile, his heart rate is 72 bpm, and his lung sounds are normal.",The probability of community acquired pneumonia (CAP) is:,"Based on this probability, you choose the following management option:","Based on a validated clinical decision rule that uses, signs, symptoms, and CRP, the patient’s likelihood of CAP is actually 2%. Based on that revised probability, what is your decision:","You feel that CAP is adequately ruled out, no CXR is needed",You order an CXR to evaluate for possible CAP,You feel that CAP is likely enough to prescribe an antibiotic without a CXR,2
1,2,"A patient presents with a 3 day history of cough productive of yellowish green sputum and a runny nose. He is not short of breath, his measured temp is 100.6 F, his heart rate is 80 bpm, and his lung exam reveals crackles.",The probability of community acquired pneumonia (CAP) is:,"Based on this probability, you choose the following management option:","Based on a validated clinical decision rule that uses, signs, symptoms, and CRP, the patient’s likelihood of CAP is actually 17%. Based on that revised probability, what is your decision:","You feel that CAP is adequately ruled out, no CXR is needed",You order an CXR to evaluate for possible CAP,You feel that CAP is likely enough to prescribe an antibiotic without a CXR,17
2,3,"A patient presents with a 3 day history of cough productive of yellowish green sputum and he has a runny nose. He is somewhat short of breath. His measured temp is 101.0 F, his heart rate is 104 bpm, and his lung exam reveals crackles.",The probability of community acquired pneumonia (CAP) is:,"Based on this probability, you choose the following management option:","Based on a validated clinical decision rule that uses, signs, symptoms, and CRP, the patient’s likelihood of CAP is actually 45%. Based on that revised probability, what is your decision:","You feel that CAP is adequately ruled out, no CXR is needed",You order an CXR to evaluate for possible CAP,You feel that CAP is likely enough to prescribe an antibiotic without a CXR,45
3,4,"A patient presents with a 3 day history of cough productive of yellowish green sputum and a runny nose. He has a measured temperature of 38.4 C (101.1 F), his heart rate is 72 bpm, and his lung sounds are normal.",The probability of community acquired pneumonia (CAP) is:,"Based on this probability, you choose the following management option:","Based on a validated clinical decision rule that uses, signs, symptoms, and CRP, the patient’s likelihood of CAP is actually 5%. Based on that revised probability, what is your decision:","You feel that CAP is adequately ruled out, no CXR is needed",You order an CXR to evaluate for possible CAP,You feel that CAP is likely enough to prescribe an antibiotic without a CXR,5
4,5,"A patient presents with a 3 day history of cough productive of yellowish green sputum and a runny nose. He is somewhat short of breath, his measured temp is 101.2 F, his heart rate is 80 bpm, and his lung exam reveals crackles.",The probability of community acquired pneumonia (CAP) is:,"Based on this probability, you choose the following management option:","Based on a validated clinical decision rule that uses, signs, symptoms, and CRP, the patient’s likelihood of CAP is actually 24%. Based on that revised probability, what is your decision:","You feel that CAP is adequately ruled out, no CXR is needed",You order an CXR to evaluate for possible CAP,You feel that CAP is likely enough to prescribe an antibiotic without a CXR,24
5,6,"A patient presents with a 3 day history of cough productive of yellowish green sputum. He denies having a runny nose. He is somewhat short of breath. His measured temp is 101.4 F, his heart rate is 106 bpm, and

In [16]:
cases['scenario'].values

array(['A patient presents with a 3 day history of cough productive of yellowish green sputum and a runny nose. He is afebrile, his heart rate is 72 bpm, and his lung sounds are normal.',
       'A patient presents with a 3 day history of cough productive of yellowish green sputum and a runny nose. He is not short of breath, his measured temp is 100.6 F, his heart rate is 80 bpm, and his lung exam reveals crackles.',
       'A patient presents with a 3 day history of cough productive of yellowish green sputum and he has a runny nose. He is somewhat short of breath. His measured temp is 101.0 F, his heart rate is 104 bpm, and his lung exam reveals crackles.',
       'A patient presents with a 3 day history of cough productive of yellowish green sputum and a runny nose. He has a measured temperature of 38.4 C (101.1 F), his heart rate is 72 bpm, and his lung sounds are normal.',
       'A patient presents with a 3 day history of cough productive of yellowish green sputum and a runny nose

In [17]:
# What is the probability of CAP
# What decision would you make? 
# Given the "true" prob. of CAP, what is your decision

In [18]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert physician estimating your confidence that a patient has community-acquired pneumonia using only the clinical presentation you're reading.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | llm

In [19]:
format_instructions = "Your response should be a SINGLE numerical probability estimate in the form of `The likelihood of CAP is [number]%.` DO NOT give a range."
manage_format_instructions = "Your response should be a single choice from (1) (2), or (3)."

In [20]:
pretest_template = "{scenario}\n\n{prob_question} Let's think step by step. First, explain your reasoning, and then give your answer. {format_instructions}"
management_template = "{management_question}.\n\n(1) {option1}\n(2) {option2}\n(3) {option3}\n\nIn addition to your probability estimate, also base your decision on your knowledge of community-acquired pneumonia and the details of this clinical scenario in particular. Let's think step by step. First, explain your reasoning, and then give your answer.\n\n{manage_format_instructions}"

true_prob_management_template = "{true_prob_management_question}\n\n(1) {option1}\n(2) {option2}\n(3) {option3}\n\nIn addition to your probability estimate, also base your decision on your knowledge of community-acquired pneumonia and the details of this clinical scenario in particular. Let's think step by step. First, explain your reasoning, and then give your answer.\n\n{manage_format_instructions}"

## Pretest Probabilities

In [21]:
chat_histories = ebell_CAP_prob_llm(chain, cases,
                                    pretest_template, management_template, true_prob_management_template, 
                                    format_instructions, manage_format_instructions,
                                    verbose=False)


Current row: 0
Current row: 1
Current row: 2
Current row: 3
Current row: 4
Current row: 5
Current row: 6
Current row: 7


In [22]:
pickle.dump(chat_histories, open(f"ebell_outputs/ebell_chat_histories_{version}.pickle", "wb" ) )

In [23]:
from ebell_llm_funcs import ebell_get_probs_from_llm, ebell_get_management_decision_from_llm, ebell_get_llm_responses


In [24]:
pretest_probs = ebell_get_probs_from_llm(chat_histories)

In [25]:
choice_map = {1 : "You feel that CAP is adequately ruled out, no CXR is needed",
2: "You order an CXR to evaluate for possible CAP",
3: "You feel that CAP is likely enough to prescribe an antibiotic without a CXR"}

In [26]:
manage_decisions, true_prob_manage_decisions = ebell_get_management_decision_from_llm(chat_histories, choice_map)

In [27]:
responses = ebell_get_llm_responses(chat_histories)

In [28]:
decisions_df = pd.DataFrame({"pretest_prob":pretest_probs, "management_decision":manage_decisions, "true_prob_management_decision":true_prob_manage_decisions})
decision_responses_df = pd.DataFrame({"pretest_llm_output":responses[0], "management_decision_llm_output":responses[1], "true_prob_management_decision_llm_output":responses[2]})

In [29]:
cases_with_gpt = pd.concat([cases, decisions_df, decision_responses_df],axis=1)

In [30]:
# with pd.option_context("display.max_colwidth", None):
#     display(cases_with_gpt.head(3))

In [31]:
cases_with_gpt.to_csv(f"ebell_outputs/all_cases_gpt4_output_{version}.csv", index=False)

In [32]:
cases_with_gpt

,note_id,scenario,probability_q,management_q,valupdate_management_q,management_option_1,management_option_2,management_option_3,true_pretest,pretest_prob,management_decision,true_prob_management_decision,pretest_llm_output,management_decision_llm_output,true_prob_management_decision_llm_output
0,1,A patient presents with a 3 day history of cou...,The probability of community acquired pneumoni...,"Based on this probability, you choose the foll...",Based on a validated clinical decision rule th...,"You feel that CAP is adequately ruled out, no ...",You order an CXR to evaluate for possible CAP,You feel that CAP is likely enough to prescrib...,2,10.0,"You feel that CAP is adequately ruled out, no ...","You feel that CAP is adequately ruled out, no ...",To estimate the probability of community-acqui...,Given the low likelihood of CAP based on the c...,With the additional information that a validat...
1,2,A patient presents with a 3 day history of cou...,The probability of community acquired pneumoni...,"Based on this probability, you choose the foll...",Based on a validated clinical decision rule th...,"You feel that CAP is adequately ruled out, no ...",You order an CXR to evaluate for possible CAP,You feel that CAP is likely enough to prescrib...,17,60.0,You order an CXR to evaluate for possible CAP,"You feel that CAP is adequately ruled out, no ...",To estimate the likelihood of community-acquir...,Given the probability estimate of 60% for the ...,With a revised probability of 17% for communit...
2,3,A patient presents with a 3 day history of cou...,The probability of community acquired pneumoni...,"Based on this probability, you choose the foll...",Based on a validated clinical decision rule th...,"You feel that CAP is adequately ruled out, no ...",You order an CXR to evaluate for possible CAP,You feel that CAP is likely enough to prescrib...,45,70.0,You order an CXR to evaluate for possible CAP,You order an CXR to evaluate for possible CAP,To estimate the likelihood that this patient h...,"Given the estimated probability of CAP at 70%,...",With a revised likelihood of CAP at 45% based ...
3,4,A patient presents with a 3 day history of cou...,The probability of community acquired pneumoni...,"Based on this probability, you choose the foll...",Based on a validated clinical decision rule th...,"You feel that CAP is adequately ruled out, no ...",You order an CXR to evaluate for possible CAP,You feel that CAP is likely enough to prescrib...,5,30.0,You order an CXR to evaluate for possible CAP,"You feel that CAP is adequately ruled out, no ...",To estimate the probability of community-acqui...,Given the probability estimate of 30% for comm...,"With a revised probability of 5%, the likeliho..."
4,5,A patient presents with a 3 day history of cou...,The probability of community acquired pneumoni...,"Based on this probability, you choose the foll...",Based on a validated clinical decision rule th...,"You feel that CAP is adequately ruled out, no ...",You order an CXR to evaluate for possible CAP,You feel that CAP is likely enough to prescrib...,24,70.0,You order an CXR to evaluate for possible CAP,You order an CXR to evaluate for possible CAP,To estimate the likelihood of community-acquir...,Given the probability estimate of 70% for comm...,With a revised probability of 24% for communit...
5,6,A patient presents with a 3 day history of cou...,The probability of community acquired pneumoni...,"Based on this probability, you choose the foll...",Based on a validated clinical decision rule th...,"You feel that CAP is adequately ruled out, no ...",You order an CXR to evaluate for possible CAP,You feel that CAP is likely enough to prescrib...,62,80.0,You order an CXR to evaluate for possible CAP,You order an CXR to evaluate for possible CAP,Community-acquired pneumonia (CAP) is often di...,Given the clinical presentation of the patient...,A 62% probability of community-acquired pneumo...
6,7,A patient presents with a 3 day history of cou...,T